# 🚀 University-1652: LPN (Local Partition Network) Model Test Pipeline

Bu notebook, University-1652 veri seti üzerinde eklediğimiz **LPN (Local Partition Network)** katmanını test etmek üzere hazırlanmıştır.
LPN katmanı, Global Average Pooling yerine görüntüyü bölgesel parçalara bölerek her parçadan bağımsız özellikler çıkarır ve bu sayede modelin bölgesel detayları (bina köşeleri, çatılar vs.) daha iyi öğrenmesini sağlar.

## 1. Ortam Kurulumu ve Veri Seti

In [ ]:
!pip install timm einops scipy

import torch
import torchvision
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Lütfen veri setini Colab'a yükleyin. Google Drive'a kopyaladıysanız aşağıdaki hücreyi kullanarak drive'ı bağlayın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ÖRNEK: Eğer proje klasörünüz drive'da ise çalışma alanını oraya ayarlayın
import os
os.chdir('/content/drive/MyDrive/University1652-Baseline/cross-view-geo-localization')
print("Aktif dizin:", os.getcwd())
# !ls -l

## 1.5. MiDaS ile Depth Map Üretimi (Opsiyonel / RGB-D Modeli İçin)

Eğer LPN modelinize 4-kanallı RGB-D yapısını dahil edecekseniz veya uydu görüntüleri üzerinden derinlik bilgisine ihtiyacınız var ise eğitim öncesi MiDaS ile bu derinlik haritalarını üretmeniz gerekmektedir. Sadece standart RGB olarak eğitecekseniz bu aşamayı atlayabilirsiniz. Midas derinlik haritalarını, `train/satellite` klasöründen okuyup `train/satellite_depth` (veyahut test için `test/query_satellite_depth`) klasörüne çıkartır.

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Varsa mevcut model ve transformu kullan, yoksa yükle
if 'midas' not in globals():
    midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    midas.to(device)
    midas.eval()

if 'transform' not in globals():
    midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
    transform = midas_transforms.small_transform

print(f"✅ MiDaS Model hazır (Device: {device})")

def iter_images(src_dir, exts=(".jpg", ".jpeg", ".png", ".bmp")):
    for root, _, files in os.walk(src_dir):
        for name in files:
            if name.lower().endswith(exts):
                yield os.path.join(root, name)

def pick_existing_dir(*candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

def load_image_any(path):
    # 1) OpenCV
    img = cv2.imread(path)
    if img is not None:
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 2) PIL
    try:
        with Image.open(path) as im:
            return np.array(im.convert("RGB"))
    except Exception:
        pass

    # 3) Bytes + OpenCV decode (özellikle özel isimlerde işe yarayabilir)
    try:
        with open(path, "rb") as f:
            buf = np.frombuffer(f.read(), dtype=np.uint8)
        img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        if img is not None:
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except Exception:
        pass

    return None

def generate_all_depth_maps(data_dir, output_root):
    """
    Hem Satellite hem de Drone görüntüleri için depth map oluşturur.
    Klasör yapısını korur ve doğrudan derinlik haritasını kaydeder.
    """
    targets = [
        # Train Seti
        ((os.path.join(data_dir, 'train', 'satellite'),), os.path.join(output_root, 'train', 'satellite_depth')),
        ((os.path.join(data_dir, 'train', 'street'), os.path.join(data_dir, 'train', 'drone')),
         os.path.join(output_root, 'train', 'street_depth')),

        # Test Seti
        ((os.path.join(data_dir, 'test', 'query_satellite'),), os.path.join(output_root, 'test', 'query_satellite_depth')),
        ((os.path.join(data_dir, 'test', 'gallery_drone'),
          os.path.join(data_dir, 'test', 'gallery_street'),
          os.path.join(data_dir, 'test', 'gallery_satellite')),
         os.path.join(output_root, 'test', 'gallery_drone_depth')),
    ]

    os.makedirs(output_root, exist_ok=True)
    print(f"🚀 Derinlik haritası üretimi başlıyor... (output_root: {output_root})")

    total_saved = 0
    total_skipped = 0

    for src_candidates, dst_dir in targets:
        src_dir = pick_existing_dir(*src_candidates)
        if not src_dir:
            print(f"⚠️ Kaynak klasör bulunamadı, atlanıyor: {src_candidates}")
            continue

        os.makedirs(dst_dir, exist_ok=True)
        img_paths = list(iter_images(src_dir))
        if not img_paths:
            print(f"⚠️ Görüntü bulunamadı: {src_dir}")
            continue

        print(f"\n📂 İşleniyor: {src_dir} -> {dst_dir} ({len(img_paths)} dosya)")

        saved = 0
        skipped = 0

        for img_path in tqdm(img_paths):
            # Çıktı yolu (aynı klasör yapısını koru, uzantıyı .png yap)
            rel_path = os.path.relpath(img_path, src_dir)
            base, _ = os.path.splitext(rel_path)
            output_path = os.path.join(dst_dir, base + ".png")
            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            # Resume (boş dosya varsa yeniden üret)
            if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
                skipped += 1
                continue

            img_rgb = load_image_any(img_path)
            if img_rgb is None:
                print(f"⚠️ Okunamadı: {img_path}")
                skipped += 1
                continue

            input_batch = transform(img_rgb).to(device)
            if input_batch.dim() == 3:
                input_batch = input_batch.unsqueeze(0)

            with torch.no_grad():
                prediction = midas(input_batch)
                prediction = torch.nn.functional.interpolate(
                    prediction.unsqueeze(1),
                    size=img_rgb.shape[:2],
                    mode="bicubic",
                    align_corners=False,
                ).squeeze()

            depth_map = prediction.cpu().numpy()
            depth_normalized = cv2.normalize(depth_map, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
            Image.fromarray(depth_normalized).save(output_path)
            saved += 1

        total_saved += saved
        total_skipped += skipped
        print(f"✅ Kaydedilen depth map sayısı: {saved} | Atlanan: {skipped}")

    print(f"\n✅ Tüm işlemler tamamlandı! Toplam kaydedilen: {total_saved}, Toplam atlanan: {total_skipped}")

# KULLANIM:
# data_root = '/content/cvpr2017_cvusa'         # Orijinal verilerin olduğu yer
# depth_output_root = '/content/cvpr2017_cvusa_depth'  # Çıktıların olacağı yer
# generate_all_depth_maps(data_root, depth_output_root)

## 2. LPN Modeli Eğitimi

Eğitim sırasında yeni eklenen parametreleri kullanacağız:
- `--pool lpn`: Global Pooling yerine LPN katmanlarını aktif eder.
- `--lpn_blocks 4`: Görüntünün kaça bölüneceğini belirler (4 parça).
- `--lpn_mode square`: Kare şeklindeki (2x2) veya `horizontal` (1x4 yatay) bölme stratejisi.
- `--batchsize 8`: LPN feature boyutunu büyüttüğü için (2048 x 4 = 8192) CUDA memory taşmalarını engellemek adına batch size düşürülmüştür.
- `--use_rgbd`: Çıkarılan derinlik haritalarını modele 4. kanal (RGB-D) olarak dahil eder.

In [ ]:
# LPN Modeli ile Eğitim (Square Mode -> 2x2 blocks + RGB-D)
!mkdir -p ./model
!python train.py \
    --name lpn_square_test \
    --data_dir ./University-1652 \
    --views 2 \
    --pool lpn \
    --lpn_blocks 4 \
    --lpn_mode square \
    --batchsize 8 \
    --lr 0.01 \
    --droprate 0.5 \
    --use_rgbd \
    --fp16

## 3. Test ve Değerlendirme (Inference)

Eğittiğimiz LPN modelinin Retrieval performansını test ediyoruz. (Modeli `--use_rgbd` ile eğittiğimiz için test aşamasında da `--use_rgbd` parametresini eklemeliyiz.)

In [ ]:
# Modeli test setinde koşturup özellik vektörlerini çıkartma
!python test.py \
    --name lpn_square_test \
    --test_dir ./University-1652/test \
    --gpu_ids 0 \
    --use_rgbd \
    --which_epoch last

In [ ]:
# Test özelliklerinden yola çıkarak mAP, Recall@1 vb metrikleri hesaplama
import scipy.io
import os

print("Rank Performansı Değerlendiriliyor...")
!python evaluate_gpu.py

## 4. (Alternatif) Horizontal LPN Eğitimi

Yukarıdaki model kare olarak bölüyordu (2x2). Alternatif olarak bina cephelerini yatay şeritler halinde öğrenen `horizontal` LPN kurabiliriz.

In [ ]:
# Horizontal LPN Edge Case + RGBD
# !python train.py \
#     --name lpn_horizontal_test \
#     --data_dir ./University-1652 \
#     --views 2 \
#     --pool lpn \
#     --lpn_blocks 4 \
#     --lpn_mode horizontal \
#     --batchsize 8 \
#     --use_rgbd \
#     --lr 0.01 \
#     --fp16